# 02 - Backtest a trained run

Replays a saved serving bundle over the run's own TEST block (out of sample: the purged split
is rebuilt from the run's config) and backtests every strategy with fees, spread, slippage,
next-open fills and stops on the bar's high/low. The strategies' confidence scale comes from
the calibration block (`artifacts/meta.json: var_scale`).

In [ ]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
STRATEGY = "enhanced_multi_horizon"
STRATEGY_PARAMS = {}            # knobs, e.g. {"min_agreement": 0.4}
RANDOM_SEEDS = 100

In [ ]:
import os
from pathlib import Path

import pandas as pd

from neural_trade.data.processor import split_arrays
from neural_trade.registries.visualizations import Visualizations
from neural_trade.serving.predictor import Predictor
from neural_trade.strategy import BacktestConfig, Bars, SignalFrame, Strategies, backtest, build_strategy

run_dir = Path(RUN_DIR) if RUN_DIR else max((p for p in Path(RUNS_DIR).iterdir() if (p / "artifacts").is_dir()), key=os.path.getmtime)
predictor = Predictor.from_artifacts(run_dir / "artifacts")
cfg = predictor.config.copy().override(CSV_PATH=CSV_PATH)
blocks = split_arrays(cfg)
test = blocks["test"]
batch = predictor.predict(test["X"], test["last_close"])
frame = batch.to_prediction_frame(predictor.bundle.pred_scale, predictor.bundle.pred_mean, y=test["y"], split="test")
bars = Bars.from_frame(blocks["df"], test["anchor_bar"])
signals = SignalFrame.build(frame, predictor.bundle.meta["var_scale"])
print(run_dir, "|", len(frame), "test bars")

## The chosen strategy

In [ ]:
res = backtest(signals, bars, build_strategy(STRATEGY, STRATEGY_PARAMS), BacktestConfig(random_seeds=RANDOM_SEEDS))
display(pd.Series(res.summary, name=res.strategy).to_frame())
display(pd.DataFrame(res.baselines).T)
Visualizations.build("plotly_trading", res, cfg, bars=bars).show()

## Every strategy, default knobs

In [ ]:
rows = {name: backtest(signals, bars, build_strategy(name), BacktestConfig(random_seeds=0), baselines=False).summary
        for name in Strategies.list_names()}
pd.DataFrame(rows).T[["n_trades", "total_return", "sharpe_net", "max_drawdown", "hit_rate", "profit_factor",
                      "exposure", "fees_paid"]]

In [ ]:
res.trades_frame().tail(20)